# CLDK, and an MCP server over it

CLDK is a Python SDK that runs static program analysis on Java, Python and TypeScript code and gives agents one API over them (Krishna et al., 2025). This notebook attaches it to the OWASP BenchmarkPython checkout HW1 used, asks the four questions an alert classifier needs, then serves those questions as MCP tools.

In [ ]:
import importlib.metadata, os, sys, subprocess
os.environ["SYSTEM_PYTHON"] = subprocess.run(["uv", "python", "find", "3.12"], capture_output=True, text=True).stdout.strip()
print(sys.executable); print("cldk", importlib.metadata.version("cldk")); print("mcp", importlib.metadata.version("mcp"))

## 1. Let's attach our analysis to CLDK

CLDK build an analysis object for the codebase, capturing various levels of program information. It happens once per codebase and is cached for subsequent runs. So we pay the cost of analysis only once.

What you get back is the analysis object where we determine the levels of analysis we want (https://codellm-devkit.info/guides/concepts/#the-ladder-cldk-builds):

1. **symbol table:** A mapping of identifiers to their definitions and scopes within the codebase. What modules and functions define which symbols and where they are used. Think of these of as the metadata. There is one per module (a file in python)
2. **call graph:** A representation of the calling relationships between functions or methods in the codebase. Which functions call which other functions and where these calls occur.
3. **program dependency graph:** A graph capturing data and control dependencies between program elements. These are typically used to understand how changes in one part of the code may affect other parts. In this case eveything is what happens inside a fuction and not across functions.
4. **system dependency graph:** A graph representing dependencies between different modules or components in the system. This is the whole system view, showing how different parts of the codebase interact with each other in terms of call control and dataflow analysis.
   
The first attach analyses 3,710 callables and takes minutes; later runs read the cache in `cldk-demo/owasp/.codeanalyzer`.

In [ ]:
import time
from cldk import CLDK
t = time.time()
an = CLDK.python(project_path="cldk-demo/owasp", analysis_level="program_dependency_graph")
print(len(an.get_callables_overview()), "callables,", f"{time.time()-t:.0f} s")

## 2. Actions on a true positive

Bandit flags B608 at line 46 of `BenchmarkTest00099.py`. 

### Step 1: Find which callable holds the line we are interested in

In [ ]:
hit = an.locate("testcode/BenchmarkTest00099.py", 46)
print("Signature of the callable: ", hit.callable.signature)

### Step 2: Get the method the body, from the signature.

In [ ]:
print(an.get_method_bodies([hit.callable.signature]).get(hit.callable.signature, None))

### Step 3: Find who calls it and what other functions it calls

Keep the handler found in Step 1 and inspect its direct callers and callees. This identifies other functions we may need to read to understand the flagged code.

In [ ]:
sig = hit.callable.signature  # handler located in Step 1

print("Who calls this handler:")
for caller in an.callers_of(sig):
    print(f"{caller.callable} -> {sig}")

print("\nWhat this handler calls:")
for callee in an.callees_of(sig):
    print(f"{sig} -> {callee.callable}")


### Step 4: does the request value reach the SQL string? 

The data-dependence graph of the handler, read by line. `resolve_value` and `flows_to_argument` resolve parameters, globals and captures, not local variables, so for a local the dependable call is `get_ddg`.

In [ ]:
import networkx as nx

def reaches(handler, src_line, dst_line):
    """Check whether data flows from src_line to dst_line inside one handler.

    Builds a directed graph from the handler's data-dependence graph (DDG),
    where each node is a source line number and each edge is a data
    dependence labelled with the variable that carries the value.

    Returns (path, variables):
      path      - list of line numbers (as strings) from src_line to dst_line
      variables - the variable carried along each hop of the path
    Returns (False, None, None) if either line is absent or no path exists.
    """
    g = nx.DiGraph()

    # Fetch the DDG of the handler (page_size is large enough for one function)
    for e in an.get_ddg(handler, page_size=500).edges:
        # Node ids look like "<...>@<line>:<col>"; keep only the line number
        src = e.src.split("@")[-1].split(":")[0]
        dst = e.dst.split("@")[-1].split(":")[0]
        # Store the variable name on the edge so we can report it per hop
        g.add_edge(src, dst, var=e.var)

    # Graph nodes are strings, so convert the requested line numbers
    s, d = str(src_line), str(dst_line)

    # Both lines must appear in the graph and be connected
    if s in g and d in g and nx.has_path(g, s, d):
        p = nx.shortest_path(g, s, d)  # shortest chain of dependences
        # Pair consecutive lines to look up the variable on each edge
        return True, p, [g[a][b]["var"] for a, b in zip(p, p[1:])]

    # No flow found
    return False, None, None

print("00099: request (31) -> sql string (46):", reaches("BenchmarkTest00099_post", 31, 46))
print("00099: request (31) -> execute (49):   ", reaches("BenchmarkTest00099_post", 31, 49))

## 3. The same actions on a false positive

`BenchmarkTest00011.py`, B608 at line 47, where the f-string has no interpolation and the value goes into the parameter tuple.

In [ ]:
print("00011: request (39) -> sql string (47):", reaches("BenchmarkTest00011_post", 39, 47))
print("00011: request (39) -> execute (50):   ", reaches("BenchmarkTest00011_post", 39, 50))

Expected: a path for 00099 to both lines; for 00011 no path to the string and a path to the call through `bar`. That is the classifier's question answered!

## 5. The server

Four tools over the same object, in the shape of lecture 5's server: `MCPServer`, `@mcp.tool`, `ToolError` for a path outside the checkout or a line in no callable.

In [11]:
%%writefile cldk-demo/server.py
import os, subprocess
from pathlib import Path
from typing import TypedDict
from mcp.server import MCPServer                                # the SDK's main server class
from mcp.server.mcpserver.exceptions import ToolError           # raised when a tool refuses
from cldk import CLDK

# Directory holding this file; the checkout to analyse lives next to it
HERE = Path(__file__).resolve().parent

# CLDK needs to know which Python interpreter to analyse with; keep a caller-provided value if set
os.environ.setdefault("SYSTEM_PYTHON", subprocess.run(["uv", "python", "find", "3.12"], capture_output=True, text=True).stdout.strip())

# The codebase we serve; resolved so the containment check in `checked` is reliable
target = (HERE / "owasp").resolve()

# Attach once at startup. This reads the cache built in section 1;
# without that cache the first start takes minutes
an = CLDK.python(project_path=str(target), analysis_level="program_dependency_graph")

# The MCP server instance that the tools below register on
mcp = MCPServer("cldk")


def checked(path: str) -> str:
    """Validate that `path` is a real file inside the target; return it relative to the target."""
    p = (target / path).resolve()  # resolve also collapses ".." segments
    # Refuse anything that escapes the checkout or is not a file
    if not p.is_relative_to(target) or not p.is_file():
        raise ToolError(f"refused: {path} is not a file of the target")
    return p.relative_to(target).as_posix()


class Located(TypedDict):
    """Result shape of the `locate` tool."""
    signature: str | None  # full signature, None at module scope
    name: str | None       # short name, None at module scope
    module: str            # module that holds the line


@mcp.tool(description="The callable that holds a line, or module scope.")
def locate(path: str, line: int) -> Located:
    # Map (file, line) to the enclosing callable
    hit = an.locate(checked(path), line)
    return {"signature": hit.callable.signature if hit.callable else None,
            "name": hit.callable.name if hit.callable else None, "module": hit.module.module_name}


@mcp.tool(description="The source text of a callable, by signature.")
def source(signature: str) -> str:
    # get_method_bodies returns a dict keyed by signature; a missing key means unknown callable
    bodies = an.get_method_bodies([signature])
    if signature not in bodies:
        raise ToolError(f"refused: no body for {signature}")
    return bodies[signature]


@mcp.tool(description="Callers of a callable by name, with file and line.")
def callers(name: str) -> list[dict]:
    # One entry per call site found in the call graph
    return [{"file": c.file, "line": c.line, "callable": c.callable} for c in an.callers_of(name)]


class Edge(TypedDict):
    """One data-dependence edge, expressed in source lines."""
    from_line: int  # line where the value is defined
    to_line: int    # line where the value is used
    var: str        # variable carrying the value


@mcp.tool(description="Data-dependence edges of the callable that holds a line: from-line, to-line, variable.")
def dataflow(path: str, line: int) -> list[Edge]:
    # Find the callable that holds the line; module scope has no DDG
    hit = an.locate(checked(path), line)
    if not hit.callable:
        raise ToolError(f"refused: line {line} of {path} is in no callable")
    # Use the full signature: a bare name like init matches 1,230 callables
    page = an.get_ddg(hit.callable.signature, page_size=2000)
    out = []
    for e in page.edges:
        try:
            # Node ids look like "<...>@<line>:<col>"; keep only the line number
            out.append({"from_line": int(e.src.split("@")[-1].split(":")[0]),
                        "to_line": int(e.dst.split("@")[-1].split(":")[0]),
                        "var": e.var})
        except ValueError:
            # Skip nodes whose id carries no numeric line (e.g. parameters, externals)
            continue
    return out


if __name__ == "__main__":
    # Serve over stdio, as the Inspector catalog in section 6 expects
    mcp.run()

Overwriting cldk-demo/server.py


## 6. The Inspector

The catalog and the command, as in lecture 5.

In [12]:
%%writefile cldk-demo/catalog.json
{
    "mcpServers": {
        "cldk": {
            "type": "stdio",
            "command": ".venv/bin/python",
            "args": ["cldk-demo/server.py"]
        }
    }
}


Overwriting cldk-demo/catalog.json


## CLDK MCP in action

In a second terminal, start the MCP Inspector with the catalog written in section 6:

`npx @modelcontextprotocol/inspector --web --catalog cldk-demo/catalog.json`

The Inspector reads `catalog.json`, launches `cldk-demo/server.py` as a child process over stdio, and shows the four tools (`locate`, `source`, `callers`, `dataflow`) in a browser UI where each can be called by hand.

**Why the cache matters.** The server calls `CLDK.python(...)` at startup. This reads the analysis cache built in section 1 (`cldk-demo/owasp/.codeanalyzer`). Without that cache, the first start re-analyses all 3,710 callables and takes minutes. The Inspector gives up on a server that takes longer than 30 seconds to answer by default, so either run section 1 first or raise the timeout.

**Changing the timeout.** In the Inspector UI, open **Configuration** and increase **Request Timeout** (milliseconds, default 10000) and, if needed, **Maximum Total Timeout** (default 60000). For example, set the request timeout to `300000` (5 minutes) so a cold start can finish. The Inspector can also be started with a larger timeout from the command line where supported. Even with a longer timeout, building the cache in section 1 first keeps startup fast.

**Walkthrough for class:**

1. `locate` with `path = testcode/BenchmarkTest00099.py` and `line = 46`. It returns the enclosing callable, `testcode.BenchmarkTest00099.init.BenchmarkTest00099_post`, and the module `BenchmarkTest00099`. This is the same result as Step 1 in section 2, now served as a tool.
2. `dataflow` with the same arguments. It returns the data-dependence edges of that handler as `from_line`, `to_line` and `var`. Following the edges from line 31 (the request value) to line 46 (the SQL string) shows that the request value reaches the query. This is the evidence for a true positive.
3. `locate` with `path = ../expectedresults-0.1.csv`. The path resolves outside the `owasp` checkout, so `checked` raises a `ToolError` and the server refuses. This shows that the tools cannot be used to read arbitrary files on the machine.

**Takeaway:** an agent can now ask the same four questions as the notebook (where is this line, what is the code, who calls it, and where does the data flow) through a standard protocol, without direct access to CLDK or the filesystem.